In [37]:
from ultralytics import YOLO
import cv2
from sort import *

In [ ]:
model = YOLO("yolov8n.yaml")
results = model.train(data = "../assets/number_plate_recognition_Yolo_EasyOCR/License_Plate_Dataset/data.yaml", epochs = 10 , device = "cuda", batch=8, imgsz = 416)

Ultralytics 8.4.41 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 8187MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../assets/number_plate_recognition_Yolo_EasyOCR/License_Plate_Dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-4, nbs=64, nms=False, opset=None

In [38]:

license_plate_detector = YOLO("runs/detect/train-4/weights/best.pt")

In [41]:
print(license_plate_detector.model)

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
    

In [42]:
coco_model = YOLO('yolov8n.pt')


In [43]:
cap = cv2.VideoCapture("../assets/number_plate_recognition_Yolo_EasyOCR/cars.mp4")



In [44]:
import string
import easyocr

# Initialize the OCR reader
reader = easyocr.Reader(['en'], gpu=True)

# Mapping dictionaries for character conversion
dict_char_to_int = {'O': '0',
                    'I': '1',
                    'J': '3',
                    'A': '4',
                    'G': '6',
                    'S': '5'}

dict_int_to_char = {'0': 'O',
                    '1': 'I',
                    '3': 'J',
                    '4': 'A',
                    '6': 'G',
                    '5': 'S'}

In [45]:
def format_license(text):
    
    license_plate_ = ''
    mapping = {0: dict_int_to_char, 1: dict_int_to_char, 4: dict_int_to_char, 5: dict_int_to_char, 6: dict_int_to_char,
               2: dict_char_to_int, 3: dict_char_to_int}
    for j in [0, 1, 2, 3, 4, 5, 6]:
        if text[j] in mapping[j].keys():
            license_plate_ += mapping[j][text[j]]
        else:
            license_plate_ += text[j]

    return license_plate_


In [46]:
def license_complies_format(text):
    
    if len(text) != 7:
        return False

    if (text[0] in string.ascii_uppercase or text[0] in dict_int_to_char.keys()) and \
       (text[1] in string.ascii_uppercase or text[1] in dict_int_to_char.keys()) and \
       (text[2] in ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'] or text[2] in dict_char_to_int.keys()) and \
       (text[3] in ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'] or text[3] in dict_char_to_int.keys()) and \
       (text[4] in string.ascii_uppercase or text[4] in dict_int_to_char.keys()) and \
       (text[5] in string.ascii_uppercase or text[5] in dict_int_to_char.keys()) and \
       (text[6] in string.ascii_uppercase or text[6] in dict_int_to_char.keys()):
        return True
    else:
        return False



In [47]:
def write_csv(results, output_path):
    
    with open(output_path, 'w') as f:
        f.write('{},{},{},{},{},{},{}\n'.format('frame_nmr', 'car_id', 'car_bbox',
                                                'license_plate_bbox', 'license_plate_bbox_score', 'license_number',
                                                'license_number_score'))

        for frame_nmr in results.keys():
            for car_id in results[frame_nmr].keys():
                print(results[frame_nmr][car_id])
                if 'car' in results[frame_nmr][car_id].keys() and \
                   'license_plate' in results[frame_nmr][car_id].keys() and \
                   'text' in results[frame_nmr][car_id]['license_plate'].keys():
                    f.write('{},{},{},{},{},{},{}\n'.format(frame_nmr,
                                                            car_id,
                                                            '[{} {} {} {}]'.format(
                                                                results[frame_nmr][car_id]['car']['bbox'][0],
                                                                results[frame_nmr][car_id]['car']['bbox'][1],
                                                                results[frame_nmr][car_id]['car']['bbox'][2],
                                                                results[frame_nmr][car_id]['car']['bbox'][3]),
                                                            '[{} {} {} {}]'.format(
                                                                results[frame_nmr][car_id]['license_plate']['bbox'][0],
                                                                results[frame_nmr][car_id]['license_plate']['bbox'][1],
                                                                results[frame_nmr][car_id]['license_plate']['bbox'][2],
                                                                results[frame_nmr][car_id]['license_plate']['bbox'][3]),
                                                            results[frame_nmr][car_id]['license_plate']['bbox_score'],
                                                            results[frame_nmr][car_id]['license_plate']['text'],
                                                            results[frame_nmr][car_id]['license_plate']['text_score'])
                            )
        f.close()


In [48]:
def get_car(license_plate, vehicle_track_ids):
    x1, y1, x2, y2, score, class_id = license_plate

    foundIt = False

    for j in range(len(vehicle_track_ids)):
        xcar1, ycar1, xcar2, ycar2, car_id = vehicle_track_ids[j]

        if (x1 > xcar1) and (y1 > ycar1) and (x2 < xcar2) and (y2 < ycar2):
            foundIt = True
            car_index = j
            break

    if foundIt:
        return vehicle_track_ids[car_index]

    return -1, -1, -1, -1, -1

In [49]:
def read_license_plate(license_plate_crop):

    detections = reader.readtext(license_plate_crop)

    for detection in detections:
        bbox, text, score = detection 
        text = text.upper().replace(" ", "")

        if license_complies_format(text):
            return format_license(text), score 



    return None , None  

In [50]:
vehicles =  [2, 3, 5, 7]
frame_nmr = -1
ret = True
mot_tracker= Sort()
results = {}


while ret:
    frame_nmr += 1
    ret, frame = cap.read()
    if ret:
        results[frame_nmr] = {}
        # detect vehicles
        detections = coco_model(frame)[0]
        detections_ = []
        for detection in detections.boxes.data.tolist():
            x1, y1, x2, y2, score, class_id = detection
            if int(class_id) in vehicles:
                detections_.append([x1, y1, x2, y2, score])

        # track vehicles
        track_ids = mot_tracker.update(np.asarray(detections_))

        # detect license plates
        license_plates = license_plate_detector(frame)[0]
        for license_plate in license_plates.boxes.data.tolist():
            x1, y1, x2, y2, score, class_id = license_plate

            # assign license plate to car
            xcar1, ycar1, xcar2, ycar2, car_id = get_car(license_plate, track_ids)

            if car_id != -1:

                # crop license plate
                license_plate_crop = frame[int(y1):int(y2), int(x1): int(x2), :]

                # process license plate
                license_plate_crop_gray = cv2.cvtColor(license_plate_crop, cv2.COLOR_BGR2GRAY)
                _, license_plate_crop_thresh = cv2.threshold(license_plate_crop_gray, 64, 255, cv2.THRESH_BINARY_INV)

                # read license plate number
                license_plate_text, license_plate_text_score = read_license_plate(license_plate_crop_thresh)

                results[frame_nmr][car_id] = {
                                                'car': {'bbox': [xcar1, ycar1, xcar2, ycar2]},
                                                'license_plate':{
                                                                    'bbox': [x1, y1, x2, y2],
                                                                    'text': license_plate_text if license_plate_text is not None else 'UNKNOWN',
                                                                    'bbox_score': score,
                                                                    'text_score': license_plate_text_score if license_plate_text_score is not None else 0
                                                                }
                                            }
# write results
write_csv(results, 'plate_info.csv')



0: 384x640 21 cars, 1 bus, 2 trucks, 26.0ms
Speed: 4.1ms preprocess, 26.0ms inference, 6.2ms postprocess per image at shape (1, 3, 384, 640)

0: 256x416 1 License_Plate, 27.7ms
Speed: 2.0ms preprocess, 27.7ms inference, 4.5ms postprocess per image at shape (1, 3, 256, 416)

0: 384x640 23 cars, 1 bus, 2 trucks, 8.2ms
Speed: 2.8ms preprocess, 8.2ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 256x416 (no detections), 25.5ms
Speed: 2.2ms preprocess, 25.5ms inference, 1.0ms postprocess per image at shape (1, 3, 256, 416)

0: 384x640 22 cars, 1 bus, 2 trucks, 8.0ms
Speed: 2.1ms preprocess, 8.0ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 256x416 2 License_Plates, 15.1ms
Speed: 1.6ms preprocess, 15.1ms inference, 2.9ms postprocess per image at shape (1, 3, 256, 416)

0: 384x640 24 cars, 1 bus, 2 trucks, 8.1ms
Speed: 2.8ms preprocess, 8.1ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 256x416 2 License_Plates, 8.7ms
Sp

In [51]:
import pandas as pd

df = pd.read_csv('plate_info.csv')
print(df.shape)
print(df.head())

(1855, 7)
   frame_nmr  car_id                                           car_bbox  \
0          0  2309.0  [751.780029296875 1369.1063232421875 1428.3668...   
1          2  2309.0  [750.9086929400235 1379.561913372499 1431.6112...   
2          3  2309.0  [743.2073967150469 1386.599758188855 1429.1217...   
3          4  2309.0  [736.5668817828073 1392.007478242252 1422.9627...   
4          5  2309.0  [734.8501188349871 1397.5636407946222 1430.409...   

                                  license_plate_bbox  \
0  [991.2223510742188 1763.555419921875 1166.0612...   
1  [947.7896728515625 1773.46240234375 1188.05053...   
2  [972.5241088867188 1812.48388671875 1180.64465...   
3  [970.3870239257812 1828.4658203125 1171.115234...   
4  [984.8507690429688 1828.3604736328125 1173.823...   

   license_plate_bbox_score license_number  license_number_score  
0                  0.371614        UNKNOWN              0.000000  
1                  0.326697        UNKNOWN              0.000000  
2

In [52]:
import csv
import numpy as np
from scipy.interpolate import interp1d


def interpolate_bounding_boxes(data):
    # Extract necessary data columns from input data
    frame_numbers = np.array([int(row['frame_nmr']) for row in data])
    car_ids = np.array([int(float(row['car_id'])) for row in data])
    car_bboxes = np.array([list(map(float, row['car_bbox'][1:-1].split())) for row in data])
    license_plate_bboxes = np.array([list(map(float, row['license_plate_bbox'][1:-1].split())) for row in data])

    interpolated_data = []
    unique_car_ids = np.unique(car_ids)
    for car_id in unique_car_ids:

        frame_numbers_ = [p['frame_nmr'] for p in data if int(float(p['car_id'])) == int(float(car_id))]
        print(frame_numbers_, car_id)

        # Filter data for a specific car ID
        car_mask = car_ids == car_id
        car_frame_numbers = frame_numbers[car_mask]
        car_bboxes_interpolated = []
        license_plate_bboxes_interpolated = []

        first_frame_number = car_frame_numbers[0]
        last_frame_number = car_frame_numbers[-1]

        for i in range(len(car_bboxes[car_mask])):
            frame_number = car_frame_numbers[i]
            car_bbox = car_bboxes[car_mask][i]
            license_plate_bbox = license_plate_bboxes[car_mask][i]

            if i > 0:
                prev_frame_number = car_frame_numbers[i-1]
                prev_car_bbox = car_bboxes_interpolated[-1]
                prev_license_plate_bbox = license_plate_bboxes_interpolated[-1]

                if frame_number - prev_frame_number > 1:
                    # Interpolate missing frames' bounding boxes
                    frames_gap = frame_number - prev_frame_number
                    x = np.array([prev_frame_number, frame_number])
                    x_new = np.linspace(prev_frame_number, frame_number, num=frames_gap, endpoint=False)
                    interp_func = interp1d(x, np.vstack((prev_car_bbox, car_bbox)), axis=0, kind='linear')
                    interpolated_car_bboxes = interp_func(x_new)
                    interp_func = interp1d(x, np.vstack((prev_license_plate_bbox, license_plate_bbox)), axis=0, kind='linear')
                    interpolated_license_plate_bboxes = interp_func(x_new)

                    car_bboxes_interpolated.extend(interpolated_car_bboxes[1:])
                    license_plate_bboxes_interpolated.extend(interpolated_license_plate_bboxes[1:])

            car_bboxes_interpolated.append(car_bbox)
            license_plate_bboxes_interpolated.append(license_plate_bbox)

        for i in range(len(car_bboxes_interpolated)):
            frame_number = first_frame_number + i
            row = {}
            row['frame_nmr'] = str(frame_number)
            row['car_id'] = str(car_id)
            row['car_bbox'] = ' '.join(map(str, car_bboxes_interpolated[i]))
            row['license_plate_bbox'] = ' '.join(map(str, license_plate_bboxes_interpolated[i]))

            if str(frame_number) not in frame_numbers_:
                # Imputed row, set the following fields to '0'
                row['license_plate_bbox_score'] = '0'
                row['license_number'] = '0'
                row['license_number_score'] = '0'
            else:
                # Original row, retrieve values from the input data if available
                original_row = [p for p in data if int(p['frame_nmr']) == frame_number and int(float(p['car_id'])) == int(float(car_id))][0]
                row['license_plate_bbox_score'] = original_row['license_plate_bbox_score'] if 'license_plate_bbox_score' in original_row else '0'
                row['license_number'] = original_row['license_number'] if 'license_number' in original_row else '0'
                row['license_number_score'] = original_row['license_number_score'] if 'license_number_score' in original_row else '0'

            interpolated_data.append(row)

    return interpolated_data


In [53]:
# Load the CSV file
with open('plate_info.csv', 'r') as file:
    reader = csv.DictReader(file)
    data = list(reader)

# Interpolate missing data
interpolated_data = interpolate_bounding_boxes(data)

# Write updated data to a new CSV file
header = ['frame_nmr', 'car_id', 'car_bbox', 'license_plate_bbox', 'license_plate_bbox_score', 'license_number', 'license_number_score']
with open('plate_info_fixed.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=header)
    writer.writeheader()
    writer.writerows(interpolated_data)

['94', '95', '96', '97', '99', '100', '101', '102', '109', '110', '113', '115', '117', '118', '120', '121', '128', '129', '130', '131', '134', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '170', '172', '173', '174', '175', '176', '177', '178', '179', '180', '181', '182', '183', '184', '185', '186', '187', '188', '189', '190', '191', '192', '193', '194', '195', '196', '197', '198', '199', '200', '201', '202', '203', '204', '205', '206', '207', '208', '209', '210', '211', '212', '213', '214', '215', '216', '217', '218', '219'] 2308
['0', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '35'] 2309
['9', '12', '15', '16', '17', '18', '19', '20', '21', '23', '24', '25'

In [54]:
results = pd.read_csv('./plate_info_fixed.csv')

print(results.shape)
print(results.head())
print(results.dtypes)
print(results['frame_nmr'].min(), results['frame_nmr'].max())

(3015, 7)
   frame_nmr  car_id                                           car_bbox  \
0         94    2308  2058.7027672566014 1079.5224465719327 2542.855...   
1         95    2308  2057.6304741493623 1084.2373199972974 2543.501...   
2         96    2308  2058.8355519555803 1086.3511019719112 2546.791...   
3         97    2308  2058.0827779533993 1089.030325168523 2547.7569...   
4         98    2308  2056.5697942804168 1090.555813910934 2548.9805...   

                                  license_plate_bbox  \
0  2135.75146484375 1382.35498046875 2340.7446289...   
1  2173.65673828125 1368.5860595703125 2382.36206...   
2  2195.543212890625 1368.188232421875 2387.58251...   
3  2178.655029296875 1384.0335693359375 2377.4653...   
4  2192.0802001953125 1385.0159301757812 2389.942...   

   license_plate_bbox_score license_number  license_number_score  
0                  0.291551        UNKNOWN              0.000000  
1                  0.473349        AF05JEO              0.331349  
2

In [55]:
import ast

import cv2
import numpy as np
import pandas as pd


def draw_border(img, top_left, bottom_right, color=(0, 255, 0), thickness=10, line_length_x=200, line_length_y=200):
    x1, y1 = top_left
    x2, y2 = bottom_right

    cv2.line(img, (x1, y1), (x1, y1 + line_length_y), color, thickness)  #-- top-left
    cv2.line(img, (x1, y1), (x1 + line_length_x, y1), color, thickness)

    cv2.line(img, (x1, y2), (x1, y2 - line_length_y), color, thickness)  #-- bottom-left
    cv2.line(img, (x1, y2), (x1 + line_length_x, y2), color, thickness)

    cv2.line(img, (x2, y1), (x2 - line_length_x, y1), color, thickness)  #-- top-right
    cv2.line(img, (x2, y1), (x2, y1 + line_length_y), color, thickness)

    cv2.line(img, (x2, y2), (x2, y2 - line_length_y), color, thickness)  #-- bottom-right
    cv2.line(img, (x2, y2), (x2 - line_length_x, y2), color, thickness)

    return img


results = pd.read_csv('./plate_info_fixed.csv')
results['frame_nmr'] = results['frame_nmr'].astype(int)
results['car_id'] = results['car_id'].astype(int)
results['license_number_score'] = results['license_number_score'].astype(float)
results['license_number'] = results['license_number'].astype(str)

# load video
video_path = '../assets/number_plate_recognition_Yolo_EasyOCR/cars.mp4'
cap = cv2.VideoCapture(video_path)
print("Video opened:", cap.isOpened())
print("FPS:", fps)
print("Width:", width)
print("Height:", height)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Specify the codec
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter('./out_fixed.mp4', fourcc, fps, (width, height))

license_plate = {}
for car_id in np.unique(results['car_id']):
    max_ = np.amax(results[results['car_id'] == car_id]['license_number_score'])
    license_plate[car_id] = {'license_crop': None,
                             'license_plate_number': results[(results['car_id'] == car_id) &
                                                             (results['license_number_score'] == max_)]['license_number'].iloc[0]}
    cap.set(cv2.CAP_PROP_POS_FRAMES, results[(results['car_id'] == car_id) &
                                             (results['license_number_score'] == max_)]['frame_nmr'].iloc[0])
    ret, frame = cap.read()

    x1, y1, x2, y2 = ast.literal_eval(results[(results['car_id'] == car_id) &
                                              (results['license_number_score'] == max_)]['license_plate_bbox'].iloc[0].replace('[ ', '[').replace('   ', ' ').replace('  ', ' ').replace(' ', ','))

    license_crop = frame[int(y1):int(y2), int(x1):int(x2), :]
    license_crop = cv2.resize(license_crop, (int((x2 - x1) * 400 / (y2 - y1)), 400))

    license_plate[car_id]['license_crop'] = license_crop


frame_nmr = -1

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# read frames
ret = True
while ret:
    ret, frame = cap.read()
    frame_nmr += 1
    if ret:
        df_ = results[results['frame_nmr'] == frame_nmr]
        if len(df_) > 0:
            print("Drawing frame:", frame_nmr, "rows:", len(df_))

        for row_indx in range(len(df_)):
            # draw car
            car_x1, car_y1, car_x2, car_y2 = ast.literal_eval(df_.iloc[row_indx]['car_bbox'].replace('[ ', '[').replace('   ', ' ').replace('  ', ' ').replace(' ', ','))
            draw_border(frame, (int(car_x1), int(car_y1)), (int(car_x2), int(car_y2)), (0, 255, 0), 25,
                        line_length_x=200, line_length_y=200)

            # draw license plate
            x1, y1, x2, y2 = ast.literal_eval(df_.iloc[row_indx]['license_plate_bbox'].replace('[ ', '[').replace('   ', ' ').replace('  ', ' ').replace(' ', ','))
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 12)

            # crop license plate
            license_crop = license_plate[df_.iloc[row_indx]['car_id']]['license_crop']

            H, W, _ = license_crop.shape

            try:
                frame[int(car_y1) - H - 100:int(car_y1) - 100,
                      int((car_x2 + car_x1 - W) / 2):int((car_x2 + car_x1 + W) / 2), :] = license_crop

                frame[int(car_y1) - H - 400:int(car_y1) - H - 100,
                      int((car_x2 + car_x1 - W) / 2):int((car_x2 + car_x1 + W) / 2), :] = (255, 255, 255)

                (text_width, text_height), _ = cv2.getTextSize(
                    license_plate[df_.iloc[row_indx]['car_id']]['license_plate_number'],
                    cv2.FONT_HERSHEY_SIMPLEX,
                    4.3,
                    17)

                cv2.putText(frame,
                            license_plate[df_.iloc[row_indx]['car_id']]['license_plate_number'],
                            (int((car_x2 + car_x1 - text_width) / 2), int(car_y1 - H - 250 + (text_height / 2))),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            4.3,
                            (0, 0, 0),
                            17)

            except Exception as e:
                print("Drawing error:", e)

        out.write(frame)
        frame = cv2.resize(frame, (1280, 720))

        # cv2.imshow('frame', frame)
        # cv2.waitKey(0)

out.release()
cap.release()

Video opened: True
FPS: 30.0
Width: 3840
Height: 2160
Drawing frame: 0 rows: 1
Drawing frame: 1 rows: 1
Drawing frame: 2 rows: 1
Drawing frame: 3 rows: 1
Drawing frame: 4 rows: 1
Drawing frame: 5 rows: 1
Drawing frame: 6 rows: 1
Drawing frame: 7 rows: 1
Drawing frame: 8 rows: 1
Drawing frame: 9 rows: 2
Drawing frame: 10 rows: 2
Drawing frame: 11 rows: 2
Drawing frame: 12 rows: 2
Drawing frame: 13 rows: 2
Drawing frame: 14 rows: 2
Drawing frame: 15 rows: 2
Drawing frame: 16 rows: 2
Drawing frame: 17 rows: 2
Drawing frame: 18 rows: 2
Drawing frame: 19 rows: 2
Drawing frame: 20 rows: 2
Drawing frame: 21 rows: 2
Drawing frame: 22 rows: 2
Drawing frame: 23 rows: 2
Drawing frame: 24 rows: 2
Drawing frame: 25 rows: 2
Drawing frame: 26 rows: 2
Drawing frame: 27 rows: 2
Drawing frame: 28 rows: 2
Drawing frame: 29 rows: 2
Drawing frame: 30 rows: 2
Drawing frame: 31 rows: 2
Drawing frame: 32 rows: 2
Drawing frame: 33 rows: 2
Drawing frame: 34 rows: 2
Drawing frame: 35 rows: 2
Drawing frame: 36 ro